# Physics-Informed Neural Networks: Methodology

This notebook presents the mathematical formulation and problem setup for applying PINNs to electromagnetic field prediction.

## Maxwell's Equations in 2D

### Governing Equations

For magnetic field prediction in 2D, we work with **Maxwell's equations** in the magnetostatic approximation {cite}`sadiku2014elements`:

**Ampère's Law (Magnetostatic)**:
$$\nabla \times \mathbf{H} = \mathbf{J}$$

**Gauss's Law for Magnetism**:
$$\nabla \cdot \mathbf{B} = 0$$

**Constitutive Relations**:
$$\mathbf{B} = \mu \mathbf{H}$$

In 2D Cartesian coordinates $(x, y)$ with only $z$-directed current density:

$$\frac{\partial H_y}{\partial x} - \frac{\partial H_x}{\partial y} = J_z$$
$$\frac{\partial B_x}{\partial x} + \frac{\partial B_y}{\partial y} = 0$$

where $B_x, B_y$ are the magnetic flux density components and $H_x, H_y$ are the magnetic field components.

### Physical Interpretation

- **Ampère's Law**: Current sources create circulating magnetic fields
- **Gauss's Law**: Magnetic field lines form closed loops (no magnetic monopoles)
- **Constitutive Relation**: Material property links B-field and H-field

### PINN Formulation

A PINN learns a function {cite}`goodfellow2016deep`:
$$\mathbf{H}(x, y; \theta) = \begin{bmatrix} H_x(x, y; \theta) \\ H_y(x, y; \theta) \end{bmatrix}$$

where $\theta$ are the neural network parameters.

The **physics loss** enforces Maxwell's equations at collocation points:

$$\mathcal{L}_{\text{physics}} = \frac{1}{N_d}\sum_{i=1}^{N_d} \left[ \left( \frac{\partial H_y}{\partial x} - \frac{\partial H_x}{\partial y} - J_z \right)^2 + \left( \frac{\partial B_x}{\partial x} + \frac{\partial B_y}{\partial y} \right)^2 \right]_i$$

where $N_d$ is the number of domain collocation points.

The **data loss** matches observed measurements:

$$\mathcal{L}_{\text{data}} = \frac{1}{N_{\text{obs}}}\sum_{j=1}^{N_{\text{obs}}} \left[ (H_x^{\text{pred}} - H_x^{\text{obs}})^2 + (H_y^{\text{pred}} - H_y^{\text{obs}})^2 \right]_j$$

## 2D Electromagnetic Problem: Magnetic Field Around Current-Carrying Wire

We'll solve the classic problem of magnetic field distribution around a current-carrying wire in 2D. This problem has an analytical solution (Biot-Savart law) {cite}`sadiku2014elements` making it perfect for PINN validation.

### Problem Setup
- **Domain**: Square region $[-1, 1] \times [-1, 1]$ (2m × 2m)
- **Wire**: Located at origin $(0, 0)$ with current $I$ flowing in $z$-direction
- **Physics**: Magnetostatic Maxwell's equations
- **Boundary**: Far-field conditions at domain edges
- **Material**: Free space with permeability $\mu_0 = 4\pi \times 10^{-7}$ H/m

### Analytical Solution (Biot-Savart Law)

For an infinite straight wire carrying current $I$ in the $z$-direction, the magnetic field at distance $r$ is {cite}`sadiku2014elements`:

$$|\mathbf{H}| = \frac{I}{2\pi r}$$

The field circulates around the wire following the right-hand rule:

$$H_x(x, y) = -\frac{I}{2\pi r^2} y$$
$$H_y(x, y) = \frac{I}{2\pi r^2} x$$

where $r = \sqrt{x^2 + y^2}$.

This analytical solution serves as **ground truth** for validating our PINN.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class ElectromagneticProblem:
    """2D electromagnetic field problem setup"""

    def __init__(self, domain_size=2.0, wire_position=(0.0, 0.0), current=1.0, mu_0=4*np.pi*1e-7):
        self.domain_size = domain_size
        self.wire_position = wire_position
        self.current = current
        self.mu_0 = mu_0  # Permeability of free space

    def analytical_solution(self, x, y):
        """Analytical solution using Biot-Savart law for infinite wire"""
        # Distance from wire
        r = np.sqrt((x - self.wire_position[0])**2 + (y - self.wire_position[1])**2)

        # Avoid singularity at wire center
        r = np.maximum(r, 1e-6)

        # Magnetic field magnitude (Biot-Savart law)
        B_magnitude = (self.mu_0 * self.current) / (2 * np.pi * r)

        # Field components (perpendicular to radial direction)
        theta = np.arctan2(y - self.wire_position[1], x - self.wire_position[0])
        Hx = -B_magnitude * np.sin(theta) / self.mu_0
        Hy = B_magnitude * np.cos(theta) / self.mu_0

        return Hx, Hy

    def generate_training_data(self, n_data=100):
        """Generate sparse training data points"""
        # Random points in domain
        x_data = np.random.uniform(-self.domain_size/2, self.domain_size/2, n_data)
        y_data = np.random.uniform(-self.domain_size/2, self.domain_size/2, n_data)

        # Get analytical solution
        Hx_data, Hy_data = self.analytical_solution(x_data, y_data)

        return x_data, y_data, Hx_data, Hy_data

    def generate_collocation_points(self, n_domain=1000, n_boundary=100):
        """Generate collocation points for physics constraints (reduced for speed)"""
        # Domain points (avoid wire singularity)
        x_domain = np.random.uniform(-self.domain_size/2, self.domain_size/2, n_domain)
        y_domain = np.random.uniform(-self.domain_size/2, self.domain_size/2, n_domain)

        # Remove points too close to wire
        r_from_wire = np.sqrt((x_domain - self.wire_position[0])**2 +
                              (y_domain - self.wire_position[1])**2)
        mask = r_from_wire > 0.1
        x_domain = x_domain[mask]
        y_domain = y_domain[mask]

        # Boundary points
        boundary_points = []

        # Top and bottom boundaries
        x_boundary = np.random.uniform(-self.domain_size/2, self.domain_size/2, n_boundary//2)
        y_boundary_top = np.ones(n_boundary//4) * self.domain_size/2
        y_boundary_bottom = -np.ones(n_boundary//4) * self.domain_size/2

        boundary_points.extend(zip(x_boundary[:n_boundary//4], y_boundary_top))
        boundary_points.extend(zip(x_boundary[n_boundary//4:], y_boundary_bottom))

        # Left and right boundaries
        y_boundary = np.random.uniform(-self.domain_size/2, self.domain_size/2, n_boundary//2)
        x_boundary_left = -np.ones(n_boundary//4) * self.domain_size/2
        x_boundary_right = np.ones(n_boundary//4) * self.domain_size/2

        boundary_points.extend(zip(x_boundary_left, y_boundary[:n_boundary//4]))
        boundary_points.extend(zip(x_boundary_right, y_boundary[n_boundary//4:]))

        x_boundary, y_boundary = zip(*boundary_points)
        x_boundary = np.array(x_boundary)
        y_boundary = np.array(y_boundary)

        return x_domain, y_domain, x_boundary, y_boundary

# Create problem instance
problem = ElectromagneticProblem(domain_size=2.0, current=1.0)
print("SUCCESS: Electromagnetic problem initialized")
print(f"Domain size: {problem.domain_size} x {problem.domain_size}")
print(f"Wire position: {problem.wire_position}")
print(f"Current: {problem.current} A")

## Visualizing the Analytical Solution

Let's visualize the ground truth magnetic field distribution to understand the problem.

In [ ]:
# Generate grid for visualization
resolution = 40
x = np.linspace(-1, 1, resolution)
y = np.linspace(-1, 1, resolution)
X, Y = np.meshgrid(x, y)

# Compute analytical solution
Hx_analytical, Hy_analytical = problem.analytical_solution(X, Y)
H_magnitude = np.sqrt(Hx_analytical**2 + Hy_analytical**2)

# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Analytical Solution: Magnetic Field Around Current-Carrying Wire', 
fontsize=14, fontweight='bold')

# H_x component
im1 = axes[0].contourf(X, Y, Hx_analytical, levels=20, cmap='RdBu_r')
axes[0].set_title('$H_x$ Component')
axes[0].set_xlabel('x (m)')
axes[0].set_ylabel('y (m)')
axes[0].plot(0, 0, 'ko', markersize=8, label='Wire')
plt.colorbar(im1, ax=axes[0], label='H_x (A/m)')

# H_y component
im2 = axes[1].contourf(X, Y, Hy_analytical, levels=20, cmap='RdBu_r')
axes[1].set_title('$H_y$ Component')
axes[1].set_xlabel('x (m)')
axes[1].set_ylabel('y (m)')
axes[1].plot(0, 0, 'ko', markersize=8, label='Wire')
plt.colorbar(im2, ax=axes[1], label='H_y (A/m)')

# Magnitude with vector field
im3 = axes[2].contourf(X, Y, H_magnitude, levels=20, cmap='viridis', alpha=0.7)
skip = 3
axes[2].quiver(X[::skip, ::skip], Y[::skip, ::skip], 
Hx_analytical[::skip, ::skip], Hy_analytical[::skip, ::skip],
color='white', alpha=0.8)
axes[2].set_title('|H| Magnitude with Vectors')
axes[2].set_xlabel('x (m)')
axes[2].set_ylabel('y (m)')
axes[2].plot(0, 0, 'ro', markersize=10, label='Wire')
plt.colorbar(im3, ax=axes[2], label='|H| (A/m)')

plt.tight_layout()
plt.show()

print(f"\n Field Statistics:")
print(f" Max |H|: {np.max(H_magnitude):.3e} A/m")
print(f" Min |H|: {np.min(H_magnitude):.3e} A/m")
print(f" Mean |H|: {np.mean(H_magnitude):.3e} A/m")

## Training Data Generation Strategy

One of the key advantages of PINNs is that they require **minimal labeled data**. We will use:

### 1. Data Points (Sparse Observations)
- Only **30-100 labeled points** from analytical solution
- Randomly sampled across the domain
- Provide anchoring for the neural network

### 2. Domain Collocation Points
- **~1000 points** where physics constraints are enforced
- No labels required—only coordinates
- Ensure Maxwell's equations are satisfied throughout domain
- Avoid wire singularity (exclude points within 0.1m of wire)

### 3. Boundary Collocation Points
- **~100 points** on domain boundaries
- Enforce far-field boundary conditions
- Match analytical solution at boundaries

This demonstrates the **data efficiency** of PINNs: with only 30 labeled points + physics knowledge, we can predict the entire field distribution.

In [ ]:
# Generate different types of training data
x_data, y_data, Hx_data, Hy_data = problem.generate_training_data(n_data=30)
x_domain, y_domain, x_boundary, y_boundary = problem.generate_collocation_points(
n_domain=1000, n_boundary=100
)

print(" Training Data Generated:")
print(f" Labeled data points: {len(x_data)}")
print(f" Domain collocation points: {len(x_domain)}")
print(f" Boundary collocation points: {len(x_boundary)}")
print(f"\n Total points for training: {len(x_data) + len(x_domain) + len(x_boundary)}")
print(f" But only {len(x_data)} are labeled!")

In [ ]:
# Visualize sampling strategy
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# Plot domain
ax.add_patch(plt.Rectangle((-1, -1), 2, 2, fill=False, edgecolor='black', linewidth=2))

# Plot different point types
ax.scatter(x_domain, y_domain, s=1, c='blue', alpha=0.3, label=f'Domain points (n={len(x_domain)})')
ax.scatter(x_boundary, y_boundary, s=20, c='green', alpha=0.6, marker='s', 
label=f'Boundary points (n={len(x_boundary)})')
ax.scatter(x_data, y_data, s=50, c='red', alpha=0.8, marker='*', 
label=f'Labeled data points (n={len(x_data)})')

# Plot wire
ax.plot(0, 0, 'ko', markersize=15, label='Current-carrying wire')

ax.set_xlabel('x (m)', fontsize=12)
ax.set_ylabel('y (m)', fontsize=12)
ax.set_title('PINN Training Data Sampling Strategy', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## Summary

In this notebook, we established:

1. **Mathematical Foundation**:
- Maxwell's equations for magnetostatics
- 2D formulation with Ampère's law and Gauss's law
- Physics loss formulation for PINNs

2. **Problem Setup**:
- Magnetic field around current-carrying wire
- Analytical solution from Biot-Savart law
- Domain and boundary specifications

3. **Data Strategy**:
- Sparse labeled data (30 points)
- Domain collocation points (1000 points, unlabeled)
- Boundary collocation points (100 points)

**Key Insight**: PINNs can learn from minimal labeled data by leveraging physics knowledge encoded in Maxwell's equations.

In the next notebook (06c), we will implement the PINN architecture and training pipeline to solve this problem.